In [ ]:
# 1- Import necessary classes and functions from the llama_index and langchain libraries.

In [1]:
# Installing LlamaIndex core library – this is like the engine of our indexing system
!pip install --upgrade llama-index

# Installing the plugin that allows LlamaIndex to talk to HuggingFace models
# this is like a translator that helps our engine understand HuggingFace language models
!pip install llama-index-llms-huggingface

# Installing additional support to access HuggingFace API-based models
# Like a remote control that lets our system fetch models directly from HuggingFace cloud
!pip install llama-index-llms-huggingface-api

# Installing extra packages required by HuggingFace models:
# transformers → for using pre-trained LLMs (like LLaMA)
# accelerate → for optimizing performance (like a speed booster for models)
# bitsandbytes → for using 8-bit quantized models (less memory, faster loading)
# sentencepiece → a tokenizer used by models like LLaMA, T5, etc.
#this line is as installing performance tuning tools and specialized kitchen knives for our ML kitchen
!pip install llama-index transformers accelerate bitsandbytes sentencepiece

# Redundant but useful: Installing transformers again explicitly to avoid version mismatch
# This is like double-checking that we have the right version of our most-used kitchen tool
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.7/261.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/

In [2]:
# 2- Set up the HuggingFace API key using an environment variable and directly for demonstration purposes

# Importing Required Classes and Setting the API Key (Environment Setup)
# Importing core modules from LlamaIndex
# These are our building blocks for loading documents, creating indexes, and querying
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, ServiceContext, PromptTemplate

# Importing the HuggingFace LLM connector – this is what allows our system to use LLMs from HuggingFace
from llama_index.llms.huggingface import HuggingFaceLLM

# Utility modules
from types import FunctionType  # (used later internally by LlamaIndex)
from llama_index.core import load_index_from_storage  # used for reloading stored indexes

# Standard libraries
import sys    # general-purpose system utilities
import os     # for handling environment variables like API keys
import time   # for adding wait/sleep during long processes

# HuggingFace Transformers tools
# These let us load and run language models
from transformers import AutoTokenizer, AutoModelForCausalLM

# Torch (PyTorch) – underlying deep learning engine
import torch

In [ ]:
# Setting up the HuggingFace API key
os.environ["HUGGING_FACE_HUB_TOKEN"] = "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"

In [4]:
# 3- Load data from this directory where we assume the documents are stored. You may adjust the path according to your data location.

In [5]:
from google.colab import files

uploaded = files.upload()


Saving 2307.14334.pdf to 2307.14334.pdf


In [6]:
# Creating the papers folder and moving the PDF inside it

import os
import shutil

# Creating the folder 'papers' if not already present
os.makedirs("papers", exist_ok=True)

# Moving the uploaded PDF into the 'papers' directory
shutil.move("2307.14334.pdf", "papers/2307.14334.pdf")

# Now our PDF is inside a folder called papers/, which is exactly what SimpleDirectoryReader expects.

'papers/2307.14334.pdf'

In [7]:
# Loading the Document Using SimpleDirectoryReader

from llama_index.core import SimpleDirectoryReader

print(" Starting the document loading process...")

# Reading the document(s) from the 'papers' directory
documents = SimpleDirectoryReader('papers').load_data()

# Confirming the number of documents loaded
print(f" Loaded {len(documents)} document(s)")

 Starting the document loading process...
 Loaded 37 document(s)


In [8]:
# Choosing an open-access HuggingFace model
model_name = "HuggingFaceH4/zephyr-7b-beta"

In [9]:
# Initializing the LLMPredictor using HuggingFaceLLM

# Initializing the LLM configuration
llm_predictor = HuggingFaceLLM(
    model_name=model_name,                 # using open-access model
    tokenizer_name=model_name,             # matching tokenizer
    context_window=3900,                   # how much text the model can consider
    max_new_tokens=256,                    # how much text the model will generate
    generate_kwargs={"temperature": 0.3},  # how creative the responses should be
    device_map="auto"                      # letting it automatically use GPU/CPU
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [20]:
# 5- Creating a ServiceContext to bundle the LLM setup ((Modern Way: Configuring Settings with LLM predictor))
# Setting the LLM Predictor (Zephyr)
from llama_index.core import Settings
Settings.llm = llm_predictor

In [22]:
# 6 - Indexing the loaded documents using the configured LLM and embeddings

from llama_index.core import VectorStoreIndex

print("Starting the indexing process...")

Starting the indexing process...


In [23]:
# Creating the index from our loaded documents
# index = VectorStoreIndex.from_documents(documents)

ValueError: 
******
Could not load OpenAI embedding model. If you intended to use OpenAI, please check your OPENAI_API_KEY.
Original error:
No API key found for OpenAI.
Please set either the OPENAI_API_KEY environment variable or openai.api_key prior to initialization.
API keys can be found or created at https://platform.openai.com/account/api-keys

Consider using embed_model='local'.
Visit our documentation for more embedding options: https://docs.llamaindex.ai/en/stable/module_guides/models/embeddings.html#modules
******

In [29]:
# 6 - Fixing embedding model to avoid OpenAI API key error


!pip install sentence-transformers

# Import required modules
from sentence_transformers import SentenceTransformer
from llama_index.core.base.embeddings.base import BaseEmbedding
from typing import List
import numpy as np
import asyncio

# Updated version with all required methods
class MyHuggingFaceEmbedding(BaseEmbedding):
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def _get_text_embedding(self, text: str) -> List[float]:
        return self.model.encode(text).tolist()

    def _get_query_embedding(self, query: str) -> List[float]:
        return self.model.encode(query).tolist()

    def _get_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts).tolist()

    async def _aget_query_embedding(self, query: str) -> List[float]:
        # Simple sync fallback for async requirement
        return self._get_query_embedding(query)


In [30]:
from llama_index.core import Settings

# Apply our custom embedding model that meets all LlamaIndex requirements
Settings.embed_model = MyHuggingFaceEmbedding()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ValueError: "MyHuggingFaceEmbedding" object has no field "model"

In [31]:
# Install sentence-transformers if needed
!pip install sentence-transformers

# Updated imports
from sentence_transformers import SentenceTransformer
from llama_index.core.base.embeddings.base import BaseEmbedding
from typing import List
from pydantic import PrivateAttr

# Final fixed version
class MyHuggingFaceEmbedding(BaseEmbedding):
    _model: SentenceTransformer = PrivateAttr()

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        super().__init__()
        self._model = SentenceTransformer(model_name)

    def _get_text_embedding(self, text: str) -> List[float]:
        return self._model.encode(text).tolist()

    def _get_query_embedding(self, query: str) -> List[float]:
        return self._model.encode(query).tolist()

    def _get_text_embeddings(self, texts: List[str]) -> List[List[float]]:
        return self._model.encode(texts).tolist()

    async def _aget_query_embedding(self, query: str) -> List[float]:
        return self._get_query_embedding(query)

In [32]:
from llama_index.core import Settings
Settings.embed_model = MyHuggingFaceEmbedding()

In [33]:
from llama_index.core import VectorStoreIndex
print(" Starting the indexing process...")
index = VectorStoreIndex.from_documents(documents)

 Starting the indexing process...
